In [5]:
import pandas as pd

# 1. 先把最新的原始数据读进来
df = pd.read_excel('BA_overseas_data.xlsx')

# 2. 随机抽取30条，不重复
sample_df = df.sample(n=30, random_state=42) 

# 3. 只保留 Job_id, JD_text 和你需要判断的技能列
cols_to_keep = ['Job_id', 'JD_text', 'Excel', 'SQL', 'ai_skill_explicit']
sample_df[cols_to_keep].to_excel('my_blind_coding.xlsx', index=False)

print("成功生成 my_blind_coding.xlsx")

成功生成 my_blind_coding.xlsx


In [3]:
import pandas as pd
from sklearn.metrics import accuracy_score, cohen_kappa_score

# 1. 读取你填的盲审表，以及A同学的全量原始表
my_coding = pd.read_excel('my_blind_coding.xlsx')
original_data = pd.read_excel('BA_overseas_data.xlsx')

# 2. 通过 Job_id 把你的表和他的表拼在一起
# 注意：我们只取需要对比的技能列，并加上后缀区分是谁填的
merged = pd.merge(my_coding[['Job_id', 'Excel', 'SQL', 'ai_skill_explicit']], 
                  original_data[['Job_id', 'Excel', 'SQL', 'ai_skill_explicit']], 
                  on='Job_id', suffixes=('_coder_B', '_coder_A'))

# 3. 开始算一致率
skills_to_check = ['Excel', 'SQL', 'ai_skill_explicit']

print("====== 验证结果 (Validation Results) ======")
for skill in skills_to_check:
    # A同学的答案
    y_A = merged[f'{skill}_coder_A']
    # 你（B同学）的答案
    y_B = merged[f'{skill}_coder_B']
    
    # 算出简单一致率 (百分比)
    acc = accuracy_score(y_A, y_B) * 100
    # 算出 Cohen's Kappa (统计学指标，越高代表一致性越好，通常 >0.6 就很好)
    kappa = cohen_kappa_score(y_A, y_B)
    
    print(f"\n技能: {skill}")
    print(f"  - 两人一致率 (Agreement Rate): {acc:.1f}%")
    print(f"  - 统计学一致性 (Cohen's Kappa): {kappa:.2f}")
    
    # 4. 找出你们俩填得不一样的地方，方便写进报告分析
    disagreements = merged[merged[f'{skill}_coder_A'] != merged[f'{skill}_coder_B']]
    if len(disagreements) > 0:
        print(f"  - 发生分歧的 Job_id: {disagreements['Job_id'].tolist()}")
    else:
        print("  - 没有分歧，完全一致！")

====== 验证结果 (Validation Results) ======

技能: Excel
  - 两人一致率 (Agreement Rate): 100.0%
  - 统计学一致性 (Cohen's Kappa): 1.00
  - 没有分歧，完全一致！

技能: SQL
  - 两人一致率 (Agreement Rate): 100.0%
  - 统计学一致性 (Cohen's Kappa): 1.00
  - 没有分歧，完全一致！

技能: ai_skill_explicit
  - 两人一致率 (Agreement Rate): 100.0%
  - 统计学一致性 (Cohen's Kappa): 1.00
  - 没有分歧，完全一致！
